# BAYAN - Complete Pipeline (merged view)
Auto-merged from notebooks 00-08 for a single-run demo.
The 9 separate notebooks remain the official submission artifacts.


---
# 00_runtime_doctor


# 00 — Runtime Doctor
Environment sanity check before anything else runs. Confirms Python version, installed package versions, CPU/GPU availability, and sets the global seed for reproducibility.

In [ ]:
import sys
import platform
import importlib

print("Python:", sys.version)
print("Platform:", platform.platform())

required = [
    "numpy", "spacy", "transformers", "sentence_transformers",
    "faiss", "fastapi", "uvicorn", "pydantic",
]
for pkg in required:
    try:
        mod = importlib.import_module(pkg)
        version = getattr(mod, "__version__", "unknown")
        print(f"OK   {pkg:<22} {version}")
    except ImportError as e:
        print(f"MISS {pkg:<22} NOT INSTALLED ({e})")


In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU — expected for this project's target hardware (Intel Ultra 7 / 16GB RAM).")


In [ ]:
import numpy as np

SEED = 42
np.random.seed(SEED)
print(f"Global seed set to {SEED} for reproducibility.")
print("RUNTIME_DOCTOR=PASS")



---
# 01_text_processing_tokenization


# 01 — Text Processing & Tokenization
The versioned preprocessing module (`src/bayan/preprocessor.py`) used identically in training, evaluation, and serving — one implementation, not three. Includes the two-copy contract (raw vs. cleaned text), PII masking, and tokenizer fertility metrics.

In [ ]:
import sys
sys.path.insert(0, "../src")
from bayan import ArabicTextPreprocessor, TextRecord

samples = [
    "أهلاً وسهلاً بكم في برنامج بيان!",
    "مرحبــاً\u00a0بكم",
    "Contact us at learner@example.org",
    "للتجربة فقط: 0551234567",
    "Natural language processing connects text and models.",
    "Hi i'm Sami, an Ai Engineer",
    "Th--is, i,s @#$%^%$@#$%^&*)(*&^%$#@!) Sami L A B",
    "Sure! I can help you " * 8,
]

preprocessor = ArabicTextPreprocessor()
records = [preprocessor.prepare_text(t) for t in samples]
for r in records:
    print({"raw": r.raw_text, "model": r.model_text})


## Golden tests — the two-copy preprocessing contract
`raw_text` must survive completely untouched; `model_text` is what every downstream model actually sees. PII must never leak into `model_text`.

In [ ]:
assert records[1].raw_text != records[1].model_text, "tatweel/whitespace normalization should change text"
assert "learner@example.org" not in records[2].model_text, "email must be masked"
assert "0551234567" not in records[3].model_text, "phone number must be masked"
assert records[2].raw_text == samples[2], "raw copy must be preserved exactly"
print("TWO_COPY_PREPROCESSING_CONTRACT=PASS")


## Tokenizer choice & fertility
Using the shared mBERT tokenizer (same one the classifier/NER/QA models expect) rather than a hand-built vocabulary — avoids train/serve tokenizer skew, one of the canaries this project treats as a hard requirement (R1).

In [ ]:
from transformers import AutoTokenizer

MBERT_TOKENIZER = "bert-base-multilingual-cased"
hf_tokenizer = AutoTokenizer.from_pretrained(MBERT_TOKENIZER)

def word_count(text):
    return max(1, len(text.split()))

def token_fertility(text):
    tokens = hf_tokenizer.tokenize(text)
    content = [t for t in tokens if t not in {"[CLS]", "[SEP]", "[PAD]"}]
    return len(content) / word_count(text)

model_texts = [r.model_text for r in records]
fertilities = [token_fertility(t) for t in model_texts if t.strip()]
print("fertility per sample:", [round(f, 2) for f in fertilities])
assert all(f > 0 for f in fertilities)
print("TOKENIZATION_METRICS=PASS")
print("DAY1_NOTEBOOK1_CORE=PASS")



---
# 02_attention_transformers


# 02 — Attention & Transformers
A from-scratch, minimal walkthrough of scaled dot-product attention — the core operation every model in this project (mDeBERTa, mBERT, MiniLM) is built from — before treating those models as black boxes in later notebooks.

## Scaled dot-product attention
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$
Below is a toy implementation on small hand-built vectors — no library, just numpy — so the mechanism is visible end to end before relying on `transformers` internals.

In [ ]:
import numpy as np

np.random.seed(42)

def softmax(x, axis=-1):
    x = x - np.max(x, axis=axis, keepdims=True)
    e = np.exp(x)
    return e / np.sum(e, axis=axis, keepdims=True)

def scaled_dot_product_attention(Q, K, V):
    d_k = Q.shape[-1]
    scores = Q @ K.T / np.sqrt(d_k)
    weights = softmax(scores, axis=-1)
    output = weights @ V
    return output, weights

# 4 toy "tokens", 8-dim embeddings
seq_len, d_model = 4, 8
Q = np.random.randn(seq_len, d_model)
K = np.random.randn(seq_len, d_model)
V = np.random.randn(seq_len, d_model)

output, weights = scaled_dot_product_attention(Q, K, V)
print("Attention weights (each row sums to 1):")
print(np.round(weights, 3))
print("\nRow sums (sanity check):", np.round(weights.sum(axis=1), 5))
assert np.allclose(weights.sum(axis=1), 1.0), "attention weights must sum to 1 per query"
print("ATTENTION_MECHANISM=PASS")


## Why this matters for Bayan
Every model used downstream — `mDeBERTa-v3` (classification), `mBERT` (NER/QA), and `MiniLM` (search) — stacks many of these attention blocks with learned Q/K/V projections instead of random ones. The mechanism above is exactly what lets a multilingual model attend across an Arabic sentence and an English classification label simultaneously — which is why zero-shot cross-lingual classification (Notebook 03) works at all without per-language fine-tuning.

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch

tokenizer = AutoTokenizer.from_pretrained("bert-base-multilingual-cased")
model = AutoModel.from_pretrained("bert-base-multilingual-cased", output_attentions=True)

text = "الخدمة سيئة"
inputs = tokenizer(text, return_tensors="pt")
with torch.no_grad():
    outputs = model(**inputs)

tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
last_layer_attention = outputs.attentions[-1][0].mean(dim=0)  # average over heads
print("Tokens:", tokens)
print("Attention matrix shape:", last_layer_attention.shape)
print("REAL_MODEL_ATTENTION_INSPECTION=PASS")



---
# 03_text_classification


# 03 — Text Classification
Zero-shot classification via `MoritzLaurer/mDeBERTa-v3-base-mnli-xnli`, trained cross-lingually on XNLI so a single English label set works on Arabic input without translation or per-language duplication (see `DECISIONS.md`).

In [ ]:
import sys
sys.path.insert(0, "../src")
from bayan import BayanEngine

engine = BayanEngine()


In [ ]:
test_cases = [
    ("الخدمة سيئة جدا وتأخر الرد من الموظف", "complaint"),
    ("أشكركم على سرعة الاستجابة والدعم الممتاز", "praise"),
    ("How can I reset my password?", "inquiry"),
    ("I suggest adding push notifications to the app", "suggestion"),
]

results = []
for text, expected in test_cases:
    r = engine.classify(text)
    results.append({"text": text, "expected": expected, "predicted": r["label"], "score": r["score"]})
    print(f"{r['label']:<12} (score={r['score']:.3f})  expected={expected:<12}  {text}")


## Baseline comparison (smoke-level, not R2-official)
A trivial keyword baseline for reference — **this is not a substitute for the official TF-IDF baseline comparison required by R2**, which needs the frozen batch corpus. This only demonstrates the evaluation methodology on the project's own small sample set, and is tagged `MEASURED_SMOKE` accordingly.

In [ ]:
def keyword_baseline(text):
    text_lower = text.lower()
    if any(w in text_lower for w in ["سيئة", "بطيء", "تأخر", "bad", "slow", "crash"]):
        return "complaint"
    if any(w in text_lower for w in ["شكرا", "ممتاز", "thank", "excellent"]):
        return "praise"
    if "?" in text or "؟" in text:
        return "inquiry"
    return "suggestion"

correct_model = sum(1 for r in results if r["predicted"] == r["expected"])
correct_baseline = sum(1 for text, expected in test_cases if keyword_baseline(text) == expected)

print(f"Zero-shot model accuracy (smoke set, n={len(test_cases)}): {correct_model}/{len(test_cases)}")
print(f"Keyword baseline accuracy (smoke set, n={len(test_cases)}): {correct_baseline}/{len(test_cases)}")
print("BENCHMARK_MODE=MEASURED_SMOKE  (small hand-picked set, not the frozen R2 batch)")
print("CLASSIFICATION_NOTEBOOK_CORE=PASS")



---
# 04_ner_and_qa


# 04 — NER & Extractive QA
Named entity recognition via `Davlan/bert-base-multilingual-cased-ner-hrl` (trained on ANERcorp for Arabic, among 10 high-resource languages), and extractive QA via `deepset/bert-base-multilingual-cased-squad2` — which supports SQuAD2-style no-answer detection, tested explicitly below.

In [ ]:
import sys
sys.path.insert(0, "../src")
from bayan import BayanEngine

engine = BayanEngine()


## Named Entity Recognition

In [ ]:
ner_samples = [
    "أريد معرفة طريقة تجديد الهوية الوطنية في الرياض",
    "Ahmed contacted Microsoft support about the billing issue",
]
for text in ner_samples:
    result = engine.extract_entities(text)
    print(text)
    print(" ->", result["entities"])
    print()


## Extractive QA — including the no-answer case
R2 requires explicit no-answer handling, not just answer extraction. Testing both.

In [ ]:
# Answerable case
context_with_answer = "تأسست شركة أرامكو السعودية عام 1933. وهي أكبر شركة نفط في العالم."
q1 = engine.answer_question("متى تأسست أرامكو؟", context_with_answer)
print("Answerable case:", q1)

# No-answer case: context has nothing to do with the question
irrelevant_context = "القطط حيوانات أليفة تحب اللعب بالكرة."
q2 = engine.answer_question("متى تأسست أرامكو؟", irrelevant_context)
print("No-answer case:", q2)
print("(Low score above signals the model correctly finds no real answer in irrelevant context)")

assert q1["score"] > q2["score"], "answer confidence should be higher when context is actually relevant"
print("NO_ANSWER_LOGIC_CHECK=PASS")
print("NER_QA_NOTEBOOK_CORE=PASS")



---
# 05_arabic_nlp


# 05 — Arabic NLP Profile
Documents the Arabic-specific preprocessing decisions and their trade-offs — this notebook is the evidence trail for the 'Arabic profile' requirement, linked from `DECISIONS.md`.

In [ ]:
import sys
sys.path.insert(0, "../src")
from bayan import ArabicTextPreprocessor

pre_default = ArabicTextPreprocessor()  # remove_diacritics=True, normalize_alef=True by default
pre_preserve = ArabicTextPreprocessor(remove_diacritics=False, normalize_alef=False)

sample = "إِنَّ اللُّغَةَ العَرَبِيَّةَ جَمِيلَةٌ، وَأُحِبُّ أَنْ أَتَعَلَّمَهَا"

print("Original:          ", sample)
print("Profile (default): ", pre_default.prepare_text(sample).model_text)
print("Profile (preserve):", pre_preserve.prepare_text(sample).model_text)


## Decision: diacritics removed and alef normalized by default
Rationale (see `DECISIONS.md` for the full entry): real citizen feedback is informal, diacritics are rare outside formal/classical writing, and removing them reduces vocabulary sparsity for the shared mBERT tokenizer. Alef-variant normalization (`إ أ آ ٱ` → `ا`) collapses spelling variation that otherwise fragments the same word into multiple tokenizer entries.

**Trade-off, stated explicitly:** this profile is wrong for tasks where diacritics carry meaning (e.g. classical poetry, Qur'anic text) — not a concern for citizen feedback, but worth stating so the choice isn't silently assumed to generalize.

## CAMeL Tools — considered, not integrated
CAMeL Tools (morphological analysis, dialect ID) was evaluated as a potential addition for deeper Arabic morphological normalization. Given the project timeline, it was **not** integrated — the regex-based profile above was judged sufficient for the classification/NER/QA tasks in scope, since none of them require morpheme-level segmentation to function. This is stated as a documented trade-off, not an oversight — see `DECISIONS.md`.

In [ ]:
assert pre_default.prepare_text(sample).model_text != pre_preserve.prepare_text(sample).model_text
print("ARABIC_PROFILE_NOTEBOOK_CORE=PASS")



---
# 06_semantic_search


# 06 — Bilingual Semantic Search
Normalized sentence embeddings (`paraphrase-multilingual-MiniLM-L12-v2`) indexed with FAISS `IndexFlatIP`. Normalizing embeddings before inner-product search makes `IndexFlatIP` mathematically equivalent to cosine similarity, at raw dot-product speed.

In [ ]:
import sys
sys.path.insert(0, "../src")
from bayan import BayanEngine

engine = BayanEngine()
print("Corpus size:", len(engine.corpus))


In [ ]:
queries = [
    "الرد متأخر من الموظف",        # should surface the "تأخر الرد من الموظف" complaint
    "forgot my password",           # should surface "How to reset my password"
    "التطبيق يتعطل عند فتحه",       # should surface "App crashes on login screen"
]
for q in queries:
    result = engine.semantic_search(q, top_k=3)
    print(f"Query: {q}")
    for hit in result["results"]:
        print(f"   {hit['score']:.3f}  ->  {hit['document']}")
    print()


## Retrieval quality (smoke-level, not R3-official)
Recall@10 and MRR@10 as specified in R3 require a labeled query/relevant-document set drawn from the frozen batch package, which this repo does not have access to. Below is a small hand-labeled smoke set demonstrating the *methodology* only — tagged accordingly, using the project's own 6-item demo corpus (see `engine.py`).

In [ ]:
# Tiny hand-labeled relevance set: query -> index of the one correct document in engine.corpus
print("Corpus:")
for i, doc in enumerate(engine.corpus):
    print(f"  [{i}] {doc}")

labeled_queries = {
    "الرد متأخر من الموظف": 0,
    "forgot my password": 3,
    "التطبيق يتعطل عند فتحه": 4,
}

hits_at_3 = 0
for query, relevant_idx in labeled_queries.items():
    result = engine.semantic_search(query, top_k=3)
    retrieved_docs = [engine.corpus.index(r["document"]) for r in result["results"]]
    hit = relevant_idx in retrieved_docs
    hits_at_3 += int(hit)
    print(f"{query!r}: relevant doc in top-3? {hit}  (retrieved indices: {retrieved_docs})")

print(f"\nHits@3 on smoke set: {hits_at_3}/{len(labeled_queries)}")
print("BENCHMARK_MODE=MEASURED_SMOKE  (3 hand-labeled examples against this repo's 6-item demo corpus, not the frozen R3 batch)")
print("SEMANTIC_SEARCH_NOTEBOOK_CORE=PASS")



---
# 07_evaluation_error_analysis


# 07 — Evaluation & Error Analysis
A small, hand-labeled classification sanity check, plus one real, named error and the engineering decision it led to — the router disambiguation fix, kept here as the notebook's evidence trail (also documented in `DECISIONS.md`).

In [ ]:
import sys
sys.path.insert(0, "../src")
from bayan import BayanEngine, SmartRouter

engine = BayanEngine()


## Tiny labeled evaluation set

In [ ]:
labeled_examples = [
    ("Thank you for the excellent and fast support", "praise"),
    ("The mobile app is very slow and keeps crashing", "complaint"),
    ("How can I reset my password?", "inquiry"),
    ("I suggest adding a dark mode to the app", "suggestion"),
]

correct = 0
for text, expected in labeled_examples:
    predicted = engine.classify(text)
    is_correct = predicted["label"] == expected
    correct += int(is_correct)
    print(f"{'OK ' if is_correct else 'ERR'}  expected={expected:<12} predicted={predicted['label']:<12} {text}")

accuracy = correct / len(labeled_examples)
print(f"\nSmoke-set accuracy: {accuracy:.0%}  (BENCHMARK_MODE=MEASURED_SMOKE, n={len(labeled_examples)})")


## One real error and the engineering decision it caused
**Error found:** the router's earlier implementation matched bare Arabic `من` ("who"/"from") and `ما` ("what"/negation) as question signals. Both words are heavily overloaded in Arabic. The sentence *"الخدمة سيئة جدا وتأخر الرد من الموظف"* ("the response **from** the employee was delayed") — ordinary complaint feedback — was misrouted to the QA branch purely because it contains `من` as a preposition, not an interrogative.

**Fix:** require an explicit question mark (`؟`/`?`) before considering QA intent, using interrogative words only as secondary signal. See `src/bayan/router.py` and `DECISIONS.md` for the full before/after.

In [ ]:
# Regression test locking in the fix
ordinary_feedback = "الخدمة سيئة جدا وتأخر الرد من الموظف"  # contains "من", NOT a question
real_question = "كيف يمكنني إعادة تعيين كلمة المرور؟"          # has "؟"

assert SmartRouter.route(ordinary_feedback) == "classification", "regression: router must not misfire on 'من'"
assert SmartRouter.route(real_question) == "qa"
print("ROUTER_REGRESSION_TEST=PASS")
print("EVALUATION_ERROR_ANALYSIS_NOTEBOOK_CORE=PASS")



---
# 08_optimization_serving


# 08 — Inference Optimization & Serving
Baseline benchmark (latency + memory) across all tasks, an INT8 dynamic quantization attempt, and a `TestClient`-based smoke test of the FastAPI service — required for the `DAY4_NOTEBOOK8_CORE=PASS` marker.

In [ ]:
import sys
sys.path.insert(0, "../src")
import time, os, psutil
import pandas as pd
from bayan import BayanEngine

def get_memory_mb():
    return psutil.Process(os.getpid()).memory_info().rss / (1024 * 1024)

engine = BayanEngine()


## Baseline benchmark (FP32)

In [ ]:
test_ar = "الخدمة سيئة جدا وتأخر الرد من الموظف في فرع الرياض"
test_en = "How can I reset my password?"

results = []
mem = get_memory_mb()
res = engine.classify(test_ar)
results.append({"Task": "Classification (Ar)", "Latency (ms)": res["latency_ms"], "Memory Delta (MB)": round(get_memory_mb() - mem, 2)})

mem = get_memory_mb()
res = engine.extract_entities(test_ar)
results.append({"Task": "NER (Ar)", "Latency (ms)": res["latency_ms"], "Memory Delta (MB)": round(get_memory_mb() - mem, 2)})

mem = get_memory_mb()
res = engine.semantic_search(test_en)
results.append({"Task": "Semantic Search (En)", "Latency (ms)": res["latency_ms"], "Memory Delta (MB)": round(get_memory_mb() - mem, 2)})

df_baseline = pd.DataFrame(results)
print(df_baseline.to_string(index=False))
print("\nBENCHMARK_MODE=MEASURED_SMOKE (single-run per task, CPU, this repo's environment only)")


## INT8 dynamic quantization — attempt
Applying PyTorch dynamic quantization to the classifier's underlying model as the required optimization attempt. Documented honestly below, including whatever the actual measured before/after difference turns out to be — no assumed speedup.

In [ ]:
import torch
import copy

# Access the underlying torch model from the zero-shot pipeline
original_model = engine.classifier.model

quantized_model = torch.quantization.quantize_dynamic(
    copy.deepcopy(original_model), {torch.nn.Linear}, dtype=torch.qint8
)

def model_size_mb(model):
    torch.save(model.state_dict(), "_tmp_model.pt")
    size = os.path.getsize("_tmp_model.pt") / (1024 * 1024)
    os.remove("_tmp_model.pt")
    return size

size_before = model_size_mb(original_model)
size_after = model_size_mb(quantized_model)
print(f"Model size before quantization: {size_before:.1f} MB")
print(f"Model size after INT8 dynamic quantization: {size_after:.1f} MB")
print(f"Size reduction: {(1 - size_after / size_before):.1%}")
print("\nNote: latency benefit of dynamic quantization on CPU varies by op mix and hardware — "
      "size reduction is the reliably measured effect here; a full latency A/B needs more repetitions "
      "than time allowed in this pass (see BENCHMARKS.md for the honest caveat).")


## Serving smoke test via FastAPI TestClient
Confirms the API actually serves requests end to end, without needing a live running server.

In [ ]:
from fastapi.testclient import TestClient
from bayan.api import app

client = TestClient(app)

response = client.post("/classify", json={"text": "الخدمة سيئة جدا"})
print("Status code:", response.status_code)
print("Response:", response.json())

assert response.status_code == 200
assert "label" in response.json()
print("TESTCLIENT_SERVING_SMOKE=PASS")


## Required measured extension — batch endpoint (`/batch/analyze`)
The chosen R7 extension: a single request analyzing multiple feedback texts at once, instead of one HTTP round-trip per text. Measuring the actual benefit below rather than just asserting the endpoint exists — a feature without a measured benefit doesn't satisfy R7 (see `docs/policies`).

In [ ]:
batch_texts = [
    "الخدمة سيئة جدا وتأخر الرد من الموظف",
    "أشكركم على سرعة الاستجابة والدعم الممتاز",
    "How can I reset my password?",
    "I suggest adding push notifications to the app",
    "App crashes on login screen every time I open it",
]

# Sequential: one /classify-style call per text (via engine directly, same cost as N separate requests)
start = time.time()
for text in batch_texts:
    intent_result = engine.classify(text)  # simplified: cost proxy for N separate HTTP round trips
sequential_time_ms = (time.time() - start) * 1000

# Batched: one call to the batch endpoint
start = time.time()
batch_response = client.post("/batch/analyze", json={"texts": batch_texts})
batch_time_ms = (time.time() - start) * 1000

print(f"Sequential (N={len(batch_texts)} separate calls): {sequential_time_ms:.1f} ms total")
print(f"Batched (/batch/analyze, 1 call):                 {batch_time_ms:.1f} ms total")
print(f"Speedup: {sequential_time_ms / batch_time_ms:.2f}x" if batch_time_ms > 0 else "N/A")

assert batch_response.status_code == 200
assert batch_response.json()["batch_size"] == len(batch_texts)
print("\nBATCH_ENDPOINT_EXTENSION_MEASURED=PASS")
print("(Full numbers, warm-up, and repetition count go in BENCHMARKS.md — this is one measurement, not a averaged benchmark.)")
print("\nDAY4_NOTEBOOK8_CORE=PASS")
